In [ ]:
import numpy as np
import tensorflow as tf
import pickle
import os

In [ ]:
DATA_DIR = "../data/processed"
MODEL_DIR = "../models"

MODEL_PATH = os.path.join(MODEL_DIR,"joint_model_final.keras")

In [ ]:
X_test = np.load(os.path.join(DATA_DIR,"X_test.npy"))
y_test = np.load(os.path.join(DATA_DIR,"y_test.npy"))

In [ ]:

# ── Custom Maxout Layer ────────────────────────────────────────────────────────
class MaxoutLayer(layers.Layer):
    """
    Maxout activation (Goodfellow et al., 2013).
    Each output unit takes the max over `num_pieces` linear projections.
    Piecewise-linear activation — expressive and works well with Dropout.
    """
    def __init__(self, units: int, num_pieces: int = 2, l2: float = 1e-4, **kwargs):
        super().__init__(**kwargs)
        self.units      = units
        self.num_pieces = num_pieces
        self.l2_val     = l2

    def build(self, input_shape):
        input_dim = int(input_shape[-1])
        reg = regularizers.l2(self.l2_val)
        self.W = self.add_weight(
            name="W", shape=(input_dim, self.units * self.num_pieces),
            initializer="glorot_uniform", regularizer=reg, trainable=True)
        self.b = self.add_weight(
            name="b", shape=(self.units * self.num_pieces,),
            initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        z = tf.matmul(inputs, self.W) + self.b
        z = tf.reshape(z, (-1, self.units, self.num_pieces))
        return tf.reduce_max(z, axis=-1)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"units": self.units, "num_pieces": self.num_pieces, "l2": self.l2_val})
        return cfg


In [ ]:
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={"MaxoutLayer": MaxoutLayer}
)

model.summary()

In [7]:
import numpy as np

# Load test data
X_test = np.load("../data/processed/X_test.npy")
y_true = np.load("../data/processed/y_test.npy")

In [10]:
class MaxoutLayer(layers.Layer):

    def __init__(self, units, num_pieces=2, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.num_pieces = num_pieces

    def build(self, input_shape):

        input_dim = int(input_shape[-1])

        self.W = self.add_weight(
            shape=(input_dim, self.units * self.num_pieces),
            initializer="glorot_uniform",
            trainable=True
        )

        self.b = self.add_weight(
            shape=(self.units * self.num_pieces,),
            initializer="zeros",
            trainable=True
        )

    def call(self, inputs):

        z = tf.matmul(inputs, self.W) + self.b
        z = tf.reshape(z, (-1, self.units, self.num_pieces))

        return tf.reduce_max(z, axis=-1)

NameError: name 'layers' is not defined

In [9]:
import os, re, pickle

MODEL_DIR = "../models"
DATA_DIR = "../data/processed"

JOINT_MODEL_PATH = os.path.join(MODEL_DIR, "joint_model_final.keras")
model = load_model(
    JOINT_MODEL_PATH,
    custom_objects={"MaxoutLayer": MaxoutLayer}
)

NameError: name 'MaxoutLayer' is not defined

In [8]:
y_prob = model.predict(X_test).flatten()

# Your paper threshold
threshold = 0.4
y_pred = (y_prob >= threshold).astype(int)

NameError: name 'model' is not defined